# Problem Statement 1: Audit and Clean the Career Dataset
**Author:** Member 1  
**Branch:** feature/data-quality  
**Dataset:** 100,000 Listings x 33 Columns  
**Deliverables:** Data Quality Audit, Date Normalization, Type Conversion & Data Cleaning Log

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Global Cleaning Log Table (Mandatory Rubric Deliverable)
cleaning_log = []

def log_audit(issue, affected_cols, findings, action_taken, records_impacted):
    cleaning_log.append({
        "Issue / Check": issue,
        "Affected Column(s)": affected_cols,
        "Findings / Diagnosis": findings,
        "Action Taken / Treatment": action_taken,
        "Records Impacted": records_impacted
    })
    print(f"[AUDIT LOG] {issue} | Columns: {affected_cols} -> {action_taken}")

In [4]:
# Exact file name from the official Calibo ZIP
CSV_PATH = "indian_tech_career_intelligence_2026.csv"

# 1. Preserve Raw Data (Mandatory Hackathon Rule)
df_raw = pd.read_csv(r"C:\Users\Lenovo\OneDrive\Desktop\india-tech-career-intelligence\data\indian_tech_career_intelligence_2026.csv")
df_clean = df_raw.copy()

print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

Raw Dataset Shape: 100,000 rows, 33 columns


In [5]:
# 1. Full Row Duplicates Audit
full_dups = df_raw.duplicated().sum()
log_audit(
    "Full Row Duplicates",
    "All Columns",
    f"Found {full_dups} duplicate records",
    "Retained unique records; verified 100% unique records",
    full_dups
)
if full_dups > 0:
    df_clean = df_clean.drop_duplicates()

# 2. Datetime Parsing (Day-Month-Year format)
for date_col in ['posted_date', 'application_deadline']:
    if date_col in df_clean.columns:
        init_nulls = df_clean[date_col].isnull().sum()
        df_clean[date_col] = pd.to_datetime(df_clean[date_col], dayfirst=True, errors='coerce')
        after_nulls = df_clean[date_col].isnull().sum()
        unparseable = after_nulls - init_nulls
        log_audit(
            f"Datetime Parsing: {date_col}",
            date_col,
            f"Initial nulls: {init_nulls}, Unparseable: {unparseable}",
            "Explicitly parsed using dayfirst=True with errors handled safely",
            unparseable
        )

# 3. Check Date Chronology: posted_date <= application_deadline
valid_dates = df_clean['posted_date'].notnull() & df_clean['application_deadline'].notnull()
inverted = (df_clean['posted_date'] > df_clean['application_deadline']) & valid_dates
log_audit(
    "Date Logic Sequence",
    "posted_date, application_deadline",
    f"{inverted.sum()} listings had posted > deadline",
    "Verified chronological ordering (posted_date <= application_deadline)",
    inverted.sum()
)

# 4. Convert career_growth_index from Percentage String to Float
if 'career_growth_index' in df_clean.columns:
    df_clean['career_growth_index_numeric'] = pd.to_numeric(
        df_clean['career_growth_index'].astype(str).str.replace('%', '', regex=False).str.strip(),
        errors='coerce'
    )
    log_audit(
        "Metric Type Conversion",
        "career_growth_index",
        "Stored as percentage text with '%' symbol",
        "Stripped '%' and converted to numeric float in 'career_growth_index_numeric'",
        len(df_clean)
    )

# 5. Missingness Disclosure (probation_period has 33,499 missing)
missing_series = df_clean.isnull().sum()
for col in missing_series[missing_series > 0].index:
    log_audit(
        "Missingness Disclosure",
        col,
        f"{missing_series[col]:,} missing values ({(missing_series[col]/len(df_clean))*100:.2f}%)",
        "Disclosed and documented missingness without arbitrary imputation",
        missing_series[col]
    )

print("\nAll audits completed for all 100,000 listings!")

[AUDIT LOG] Full Row Duplicates | Columns: All Columns -> Retained unique records; verified 100% unique records
[AUDIT LOG] Datetime Parsing: posted_date | Columns: posted_date -> Explicitly parsed using dayfirst=True with errors handled safely
[AUDIT LOG] Datetime Parsing: application_deadline | Columns: application_deadline -> Explicitly parsed using dayfirst=True with errors handled safely
[AUDIT LOG] Date Logic Sequence | Columns: posted_date, application_deadline -> Verified chronological ordering (posted_date <= application_deadline)
[AUDIT LOG] Metric Type Conversion | Columns: career_growth_index -> Stripped '%' and converted to numeric float in 'career_growth_index_numeric'
[AUDIT LOG] Missingness Disclosure | Columns: probation_period -> Disclosed and documented missingness without arbitrary imputation

All audits completed for all 100,000 listings!


In [6]:
# 1. Export Audited Data Cleaning Log (Mandatory Deliverable)
df_log = pd.DataFrame(cleaning_log)
df_log.to_csv("data_cleaning_log.csv", index=False)
print("Saved: data_cleaning_log.csv")

# 2. Export Cleaned CSV for Teammates
CLEANED_OUTPUT_PATH = r"data\india_tech_career_cleaned.csv"
df_clean.to_csv(CLEANED_OUTPUT_PATH, index=False)
print(f"Saved: {CLEANED_OUTPUT_PATH}")

# Display the Audit Log Table directly in the notebook
df_log

Saved: data_cleaning_log.csv
Saved: data\india_tech_career_cleaned.csv


,Issue / Check,Affected Column(s),Findings / Diagnosis,Action Taken / Treatment,Records Impacted
0,Full Row Duplicates,All Columns,Found 0 duplicate records,Retained unique records; verified 100% unique ...,0
1,Datetime Parsing: posted_date,posted_date,"Initial nulls: 0, Unparseable: 0",Explicitly parsed using dayfirst=True with err...,0
2,Datetime Parsing: application_deadline,application_deadline,"Initial nulls: 0, Unparseable: 0",Explicitly parsed using dayfirst=True with err...,0
3,Date Logic Sequence,"posted_date, application_deadline",0 listings had posted > deadline,Verified chronological ordering (posted_date <...,0
4,Metric Type Conversion,career_growth_index,Stored as percentage text with '%' symbol,Stripped '%' and converted to numeric float in...,100000
5,Missingness Disclosure,probation_period,"33,499 missing values (33.50%)",Disclosed and documented missingness without a...,33499
